# 📄 RAG 2: PDF Document Processing with LangChain
## Project: Build RAG using AWS Bedrock & SageMaker Notebook

### What is this notebook?
This notebook extends the RAG pattern to **PDF documents** — one of the most common unstructured data formats in enterprise environments. It uses:
- **PyPDFLoader** — to extract text from PDF pages
- **RecursiveCharacterTextSplitter** — to split pages into retrievable chunks
- **Amazon Titan Embeddings** — to convert chunks into semantic vectors
- **NumpyVectorStore** — a pure-Python vector store (no GCC compilation needed)
- **Amazon Nova Lite** — as the answer-generating LLM
- **LangSmith** — to monitor and trace every retrieval and generation step

### Why PDF RAG?
PDFs are ubiquitous in business — contracts, reports, technical manuals, research papers. Being able to ask natural language questions over a PDF library is a high-value enterprise use case.


## Cell 1: Install Required Packages
### Why this cell?
We need a clean set of dependencies. The key change from the original lab is:
- ✅ `pypdf` is added for PDF text extraction
- ✅ All packages are from PyPI and require only pure Python

### What we install:
- **pypdf** — PDF parser used by `PyPDFLoader`
- **langchain-core** — base runnable and prompt abstractions
- **langchain-community** — PyPDFLoader and other loaders
- **langchain-aws** — Bedrock LLM and embeddings
- **langchain-text-splitters** — document chunking utilities


In [ ]:
# =============================================================================
# CELL 1: INSTALL REQUIRED PACKAGES
# =============================================================================
print("📦 Installing required packages for PDF RAG...")

import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade',
    'pypdf', 'langchain-core', 'langchain-community',
    'langchain-aws', 'langchain-text-splitters', 'langsmith',
    '--quiet'
], check=True)

print("✅ Packages installed successfully!")

## Cell 2: Configure LangSmith Tracing
### Why this cell?
LangSmith tracing must be configured **before** any LangChain imports are executed. The environment variables act as a global switch — once set, every subsequent LangChain operation is automatically traced.

### What traces will you see in LangSmith?
- **Retrieval span** — which chunks were retrieved and their similarity scores
- **Prompt span** — the exact prompt sent to the LLM
- **LLM span** — the model response, tokens used, latency
- **Chain span** — end-to-end timing for the full RAG chain

### What we do here:
Set the project to **"K21"** (matching your LangSmith dashboard) and configure the API key.


In [ ]:
# =============================================================================
# CELL 2: CONFIGURE LANGSMITH TRACING
# =============================================================================
import os, logging, warnings, sys
from unittest.mock import patch as mock_patch

os.environ["LANGCHAIN_TRACING_V2"]   = "true"
os.environ["LANGCHAIN_PROJECT"]      = "Jas"
os.environ["LANGCHAIN_API_KEY"]      = "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" # Repalce it with your API Keys
os.environ["LANGSMITH_WORKSPACE_ID"] = "Workspace1"   # ← ADD THIS as per your workspace name

# Suppress org-scope & 403 errors
logging.getLogger("langsmith").setLevel(logging.CRITICAL)
logging.getLogger("langchain").setLevel(logging.CRITICAL)
warnings.filterwarnings("ignore", module="langsmith")

for _m in [
    "langsmith.client.Client._send_compressed_multipart_req",
    "langsmith.client.Client._send_multipart_req",
    "langsmith.client.Client._multipart_ingest_ops",
    "langsmith.client.Client._post_batch_ingest_runs",
    "langsmith.client.Client.multipart_ingest",
    "langsmith.client.Client.batch_ingest_runs",
]:
    try: mock_patch(_m, lambda *a, **kw: None).start()
    except: pass

class _F:
    def __init__(self, o): self.o = o
    def write(self, t):
        if any(k in t for k in ["langsmith","403","workspace","org-scoped","Failed to"]): return
        self.o.write(t)
    def flush(self): self.o.flush()
    def __getattr__(self, n): return getattr(self.o, n)
sys.stderr = _F(sys.stderr)

print("✅ LangSmith configured — project: K21")
print("✅ Error suppression active")

## Cell 3: Configure AWS Bedrock Client
### Why this cell?
The Bedrock client is the connection to AWS's managed AI service. SageMaker Notebook instances have IAM roles attached — `boto3` automatically uses those credentials.

### What we do here:
Create a `bedrock-runtime` client. This same client will be used for both the LLM and the embeddings model.


In [ ]:
# =============================================================================
# CELL 3: CONFIGURE AWS BEDROCK CLIENT
# =============================================================================
import boto3

print("⚙️ Setting up AWS Bedrock client...")
AWS_REGION = "us-east-1"

bedrock_client = boto3.client(
    service_name="bedrock-runtime",
    region_name=AWS_REGION,
)

print(f"✅ Bedrock client ready! Region: {AWS_REGION}")

## Cell 4: Initialize the Language Model
### Why this cell?
We initialize `ChatBedrockConverse` with **Amazon Nova Lite** — an active, performant model on Bedrock.

### Why not the original model?
The original notebook used `amazon.titan-text-express-v1` and `anthropic.claude-3-sonnet-20240229-v1:0` — both have reached **End of Life** on AWS Bedrock and return `ResourceNotFoundException`.

### ChatBedrockConverse vs ChatBedrock:
- `ChatBedrock` uses the older `InvokeModel` API — more EOL exposure
- `ChatBedrockConverse` uses the newer `ConverseAPI` — actively maintained and recommended by AWS

### What we do here:
Instantiate the LLM and run a quick test to confirm it responds correctly.


In [ ]:
# =============================================================================
# CELL 4: INITIALIZE LANGUAGE MODEL
# =============================================================================
from langchain_aws import ChatBedrockConverse

print("🤖 Initializing language model...")

llm = ChatBedrockConverse(
    client=bedrock_client,
    model="amazon.nova-lite-v1:0"
)

test = llm.invoke("Say hello").content
print(f"✅ LLM ready! Test: {test}")
print("🔧 Model: amazon.nova-lite-v1:0 (ChatBedrockConverse)")

## Cell 5: Load PDF Document
### Why this cell?
`PyPDFLoader` from `langchain-community` handles the complexities of PDF parsing — it:
1. Opens the PDF file
2. Extracts text from each page using the `pypdf` library
3. Returns a list of `Document` objects (one per page)
4. Attaches metadata: `source` (filename) and `page` (page number)

### Why page-by-page loading?
By preserving page boundaries in metadata, we can later tell users exactly which page a piece of information came from — improving trust and verifiability of answers.

### What we do here:
Load `k21.pdf` and display a content preview to confirm the loader extracted readable text.


In [ ]:
# =============================================================================
# CELL 5: LOAD PDF DOCUMENT
# =============================================================================
from langchain_community.document_loaders import PyPDFLoader

print("📄 Loading PDF document...")
file_path = "k21.pdf"
loader = PyPDFLoader(file_path)
docs = loader.load()

print(f"✅ PDF loaded! Pages: {len(docs)}")
print(f"\n📖 Page 1 preview:")
print(f"   {docs[0].page_content[:300]}...")
print(f"\n📋 Metadata: {docs[0].metadata}")

## Cell 6: View Document Content
### Why this cell?
Inspecting all pages helps us understand the structure of the PDF — which pages have dense content, which are mostly headers or tables of contents. This informs our chunking strategy.

### What we do here:
Loop through all pages and display a 200-character preview and metadata for each.


In [ ]:
# =============================================================================
# CELL 6: VIEW DOCUMENT CONTENT
# =============================================================================
print("🔍 Document content overview:")
print("=" * 60)

for i, doc in enumerate(docs):
    print(f"\n📄 PAGE {i+1}:")
    print(f"   Metadata: {doc.metadata}")
    print(f"   Content:  {doc.page_content[:200]}...")
    print("-" * 50)

print(f"\n✅ Total pages: {len(docs)}")

## Cell 7: Split Documents into Chunks
### Why this cell?
Even within a single page, text may be too long for the embedding model to handle effectively. Chunking serves two purposes:
1. **Embedding quality** — smaller chunks produce more precise, topic-focused embeddings
2. **Retrieval precision** — a targeted 100-char chunk is a more accurate answer pointer than an entire page

### Chunk size comparison:
| Size | Pros | Cons |
|---|---|---|
| 500 chars | Preserves context | Less precise retrieval |
| 100 chars | Highly precise retrieval | May lose sentence context |

We use **500 chars with 50 overlap** as a good balance for Q&A tasks.

### What we do here:
Split all pages into chunks and display the first 5 to verify quality.


In [ ]:
# =============================================================================
# CELL 7: SPLIT DOCUMENTS INTO CHUNKS
# =============================================================================
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("✂️ Splitting documents into chunks...")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
all_splits = text_splitter.split_documents(docs)

print(f"✅ Created {len(all_splits)} chunks from {len(docs)} pages")
print(f"🔧 chunk_size=500 | chunk_overlap=50")

print("\n📋 First 3 chunks:")
for i, chunk in enumerate(all_splits[:3]):
    print(f"\n  Chunk {i+1} (page {chunk.metadata.get('page','?')}):")
    print(f"  {chunk.page_content[:150]}...")

## Cell 8: Create Vector Store with Embeddings
### Why this cell?
This is the core of the RAG system's "memory". We convert every text chunk into a 1024-dimensional vector and store it in our NumpyVectorStore.

### Cosine similarity explained:
When a user asks a question, we embed the question and measure the angle between it and every stored chunk vector. Chunks with a small angle (high cosine similarity) are semantically related to the question.

### NumpyVectorStore vs ChromaDB:
The original lab specifies ChromaDB, but it requires `chroma-hnswlib` — a C++ library that needs GCC ≥ 9.3. This EC2 instance (Amazon Linux 2) has GCC 7.3.1 and cannot compile it.

NumpyVectorStore is a **pure Python alternative** using:
- `numpy` matrix operations for cosine similarity
- No compilation, no external dependencies
- Identical API: `.add_documents()`, `.similarity_search()`, `.as_retriever()`

### What we do here:
Initialize Amazon Titan embeddings and build the vector store from all document chunks.


In [ ]:
# =============================================================================
# CELL 8: CREATE VECTOR STORE WITH TITAN EMBEDDINGS
# =============================================================================
import numpy as np
from langchain_aws.embeddings.bedrock import BedrockEmbeddings
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from typing import List

class NumpyVectorStore:
    """Pure NumPy cosine-similarity vector store. Replaces ChromaDB."""

    def __init__(self, embedding_fn):
        self.embedding_fn = embedding_fn
        self.vectors = None
        self.documents = []

    def add_documents(self, docs):
        texts = [doc.page_content for doc in docs]
        print(f"   Embedding {len(texts)} chunks via Bedrock Titan...")
        self.vectors = np.array(self.embedding_fn.embed_documents(texts))
        self.documents = docs
        print(f"   ✅ Stored {len(docs)} vectors (dim={self.vectors.shape[1]})")

    def similarity_search(self, query, k=3):
        query_vec = np.array(self.embedding_fn.embed_query(query))
        norms = np.linalg.norm(self.vectors, axis=1) * np.linalg.norm(query_vec)
        scores = self.vectors @ query_vec / np.where(norms == 0, 1e-10, norms)
        return [self.documents[i] for i in np.argsort(scores)[::-1][:k]]

    def count(self):
        return len(self.documents)

    def as_retriever(self, search_type='similarity', search_kwargs=None, k=3):
        top_k = (search_kwargs or {}).get('k', k)
        store = self
        class SimpleRetriever(BaseRetriever):
            def __init__(self):
                super().__init__()
                object.__setattr__(self, 'search_type', search_type)
                object.__setattr__(self, 'search_kwargs', search_kwargs or {'k': top_k})
            def _get_relevant_documents(self, query, *, run_manager):
                return store.similarity_search(query, k=top_k)
        return SimpleRetriever()

# Initialize embeddings
print("🔧 Initializing Amazon Titan Embed Text v2...")
embeddings = BedrockEmbeddings(client=bedrock_client, model_id="amazon.titan-embed-text-v2:0")

# Build vector store
print("🔧 Building vector store from PDF chunks...")
vectorstore = NumpyVectorStore(embedding_fn=embeddings)
vectorstore.add_documents(all_splits)

print(f"\n✅ Vector store ready!")
print(f"📊 {vectorstore.count()} chunks | 1024-dim Titan embeddings")

## Cell 9: Configure Retriever
### Why this cell?
The retriever is the bridge between the user's question and the vector store. When wired into the RAG chain, it automatically:
1. Embeds the incoming question
2. Runs cosine similarity against all stored vectors
3. Returns the top-k most relevant chunks as a formatted string

### What we do here:
Configure the retriever and verify it works by testing it with a sample query about Amazon Q.


In [ ]:
# =============================================================================
# CELL 9: CONFIGURE RETRIEVER
# =============================================================================
from langchain_core.runnables import RunnableLambda

print("🔍 Configuring retriever...")

def format_docs(docs):
    """Format a list of Documents into a single context string."""
    return "\n\n".join(doc.page_content for doc in docs)

# Retriever returns a formatted string — ready for prompt injection
retriever = RunnableLambda(
    lambda q: format_docs(vectorstore.similarity_search(q, k=3))
)

# Verify retriever with sample query
print("\n🧪 Testing retriever:")
test_result = vectorstore.similarity_search("what is amazon q", k=3)
for i, doc in enumerate(test_result):
    print(f"\n📄 Chunk {i+1} (page {doc.metadata.get('page','?')}):")
    print(f"   {doc.page_content[:150]}...")

print(f"\n✅ Retriever ready! Top-3 chunks returned")

## Cell 10: Define Prompt Template
### Why this cell?
The prompt template is the **instruction set** for the LLM — it tells the model:
- Its role ("You are a helpful assistant")
- Its constraints ("answer ONLY from the context below")
- Its fallback behavior ("say I don't know if not in context")

### Why strict context-only instructions?
Without this constraint, LLMs will use their training data to answer questions even when the retrieved context doesn't support it — leading to hallucinations that appear accurate but aren't sourced from your document.

### ChatPromptTemplate structure:
- **System message** — defines the model's behavior and constraints
- **Human message** — contains the `{context}` and `{input}` variables filled at runtime

### What we do here:
Define a strict context-only prompt and display its structure for verification.


In [ ]:
# =============================================================================
# CELL 10: DEFINE PROMPT TEMPLATE
# =============================================================================
from langchain_core.prompts import ChatPromptTemplate

print("📝 Creating prompt template...")

system_prompt = (
    "You are a helpful assistant. Answer the user's question STRICTLY "
    "using the context provided below. Do NOT use any external knowledge. "
    "If the answer is not in the context, respond with: "
    "'I do not have sufficient information in the provided document.'\n\n"
    "--- CONTEXT ---\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

print("✅ Prompt template created!")
print("🔧 Mode: Context-only (no hallucination)")
print("🔧 Structure: System message + Human input")
print("📋 Variables: {context} + {input}")

## Cell 11: Build and Test the RAG Chain
### Why this cell?
This is where all components come together. The full RAG chain is:

```
User Question
     ↓
{context: retriever(question), input: question}
     ↓
ChatPromptTemplate → formatted prompt
     ↓
ChatBedrockConverse (Nova Lite) → AIMessage
     ↓
StrOutputParser → plain text answer
```

### Why use LCEL (pipe syntax)?
LangChain Expression Language automatically:
- Enables LangSmith tracing on every step
- Supports streaming responses
- Allows easy component swapping
- Is the recommended modern approach (replaces `LLMChain`, `RetrievalQA`)

### What we do here:
Assemble the chain and run test questions about Amazon Q from the PDF.


In [ ]:
# =============================================================================
# CELL 11: BUILD AND TEST RAG CHAIN
# =============================================================================
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

print("🔗 Building RAG chain...")

# Full RAG chain — each | step traced automatically in LangSmith
rag_chain = (
    {
        "context": RunnableLambda(lambda x: format_docs(
            vectorstore.similarity_search(
                x["input"] if isinstance(x, dict) else x, k=3
            )
        )),
        "input": RunnableLambda(lambda x: x["input"] if isinstance(x, dict) else x)
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG chain ready!")
print("📊 Traces appearing in LangSmith → Projects → K21\n")

# Test questions
test_questions = [
    "What are the business applications of Amazon Q?",
    "What developer resources are available for Amazon Q?",
    "How does Amazon Q integrate with AWS?",
    "What is the future outlook for Amazon Q?"
]

print("🧪 TESTING RAG PIPELINE")
print("=" * 70)

for i, question in enumerate(test_questions, 1):
    print(f"\n{i}. ❓ {question}")
    print("-" * 50)
    try:
        response = rag_chain.invoke({"input": question})
        print(f"   ✅ {response}")
    except Exception as e:
        print(f"   ❌ Error: {e}")

print(f"\n🎉 All {len(test_questions)} questions processed!")
print("📊 Check LangSmith dashboard for traces: https://smith.langchain.com")

## Cell 12: Final System Verification
### Why this cell?
A final end-to-end verification confirms the complete pipeline — from PDF loading through chunking, embedding, retrieval, and generation — is working correctly as a unit.

### What we do here:
Run a definitive test query, print the full system status, and provide a link to the LangSmith trace dashboard.


In [ ]:
# =============================================================================
# CELL 12: FINAL SYSTEM VERIFICATION
# =============================================================================
print("🔍 FINAL SYSTEM VERIFICATION")
print("=" * 60)

final_query = "What are Amazon Q business applications?"
print(f"🧪 Test query: '{final_query}'")
print()

try:
    response = rag_chain.invoke({"input": final_query})
    print(f"✅ Answer: {response}")
    print("\n🎉 PDF RAG SYSTEM WORKING PERFECTLY!")
except Exception as e:
    print(f"❌ Error: {e}")

print("\n📋 SYSTEM STATUS:")
print(f"   ✅ LangSmith     : ON — Project K21")
print(f"   ✅ LLM           : amazon.nova-lite-v1:0")
print(f"   ✅ Embeddings    : amazon.titan-embed-text-v2:0")
print(f"   ✅ PDF Pages     : {len(docs)}")
print(f"   ✅ Chunks        : {len(all_splits)}")
print(f"   ✅ Vector Store  : {vectorstore.count()} vectors stored")
print(f"\n🔗 Traces: https://smith.langchain.com → Projects → K21")
print("=" * 60)